# Chapter 3 Tutorial: Bernoulli Processes and Sums of Independent Random Variables

This notebook is a guided tutorial for Chapter 3 of Çınlar, covering:

1. Bernoulli processes
2. Number of successes $N_n$
3. Binomial distribution and independent increments
4. Conditional expectation in Bernoulli processes
5. Times of successes $T_k$
6. Geometric and negative-binomial distributions
7. Renewal/sum processes $Z_n = Y_1 + \cdots + Y_n$
8. Weak and strong laws of large numbers

The emphasis is on the concepts, proofs, and examples in the chapter, with simulations and plots to build intuition.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

rng = np.random.default_rng(7)

def simulate_bernoulli(p, n, paths=1):
    """Return an array of shape (paths, n) with Bernoulli(p) trials."""
    return (rng.random((paths, n)) < p).astype(int)

def binom_pmf(n, k, p):
    if k < 0 or k > n:
        return 0.0
    return math.comb(n, k) * (p ** k) * ((1-p) ** (n-k))

def geom_pmf(m, p):
    # waiting time until first success, support m = 1, 2, ...
    return ((1-p) ** (m-1)) * p

## 1. Bernoulli process

A **Bernoulli process** is a sequence of random variables

$$
X_1, X_2, X_3, \ldots
$$

such that:

1. each $X_n$ takes only the values $0$ and $1$;
2. $P(X_n = 1) = p$ and $P(X_n = 0) = q = 1-p$ for every $n$;
3. $X_1, X_2, \ldots$ are independent.

Think of $X_n = 1$ as a **success** and $X_n = 0$ as a **failure**.

Mental model: a Bernoulli process is the mathematical abstraction of repeating the same yes/no experiment independently under unchanged conditions.

Examples from the chapter:

- assembly-line inspection: defective or not defective;
- drivers choosing right or left at a fork;
- bearings meeting or failing specification limits.

### Basic facts

For one Bernoulli trial $X$:

$$
E[X] = p,
$$

because $X$ is $1$ with probability $p$ and $0$ with probability $q$.

Also,

$$
X^2 = X,
$$

because $0^2=0$ and $1^2=1$. Therefore

$$
E[X^2] = E[X] = p,
$$

and

$$
\operatorname{Var}(X) = E[X^2] - (E[X])^2 = p - p^2 = p(1-p)=pq.
$$

The probability generating function is

$$
E[\alpha^X] = q + \alpha p.
$$

In [ ]:
p = 0.62
q = 1-p
x = np.array([0, 1])
pmf = np.array([q, p])
EX = (x * pmf).sum()
EX2 = ((x**2) * pmf).sum()
VarX = EX2 - EX**2
EX, VarX, p*q

### Example: bearings within specification

Suppose bearing diameters are normally distributed with mean $3$ and standard deviation $0.002$.

The bearing is accepted if it lies within the interval

$$
[2.994, 3.006].
$$

Let

$$
X_n = I_{[2.994,3.006]}(Y_n),
$$

where $Y_n$ is the diameter of bearing $n$.

Then $X_n=1$ means the bearing meets specifications. Since

$$
\frac{2.994-3}{0.002} = -3, \qquad \frac{3.006-3}{0.002}=3,
$$

we get

$$
P(X_n=1)=P(-3\le Z\le 3)\approx 0.9974.
$$

In [ ]:
# Estimate P(-3 <= Z <= 3) by simulation
z = rng.standard_normal(2_000_000)
p_accept = np.mean((-3 <= z) & (z <= 3))
p_accept

## 2. Number of successes

Define

$$
N_0 = 0,
$$

and for $n\ge 1$,

$$
N_n = X_1 + X_2 + \cdots + X_n.
$$

So $N_n$ is the number of successes in the first $n$ trials.

This is the fundamental counting process associated with a Bernoulli process.

### Expectation and variance of $N_n$

Because expectation is linear,

$$
E[N_n] = E[X_1 + \cdots + X_n]
       = E[X_1] + \cdots + E[X_n]
       = np.
$$

Because the trials are independent, variances add:

$$
\operatorname{Var}(N_n)
= \operatorname{Var}(X_1)+\cdots+\operatorname{Var}(X_n)
= npq.
$$

Also,

$$
E[N_n^2] = \operatorname{Var}(N_n) + (E[N_n])^2
= npq + n^2p^2.
$$

In [ ]:
p = 0.3
n = 50
paths = 200_000
X = simulate_bernoulli(p, n, paths)
Nn = X.sum(axis=1)
Nn.mean(), n*p, Nn.var(), n*p*(1-p)

## 3. The binomial distribution

For $k=0,1,\ldots,n$,

$$
P(N_n=k)=\binom{n}{k}p^kq^{n-k}.
$$

This is the **binomial distribution**.

Mental model: to have exactly $k$ successes in $n$ trials:

1. choose which $k$ of the $n$ positions are successes: $\binom nk$ choices;
2. each chosen pattern has probability $p^kq^{n-k}$;
3. add over the mutually exclusive patterns.

### Proof by Pascal recursion

Let $N_n$ be the number of successes in the first $n$ trials. Since

$$
N_{n+1}=N_n+X_{n+1},
$$

we have, for each $k$,

$$
P(N_{n+1}=k)
= pP(N_n=k-1)+qP(N_n=k).
$$

This says: to have $k$ successes after $n+1$ trials, either:

- there were $k-1$ successes before and the new trial succeeds; or
- there were $k$ successes before and the new trial fails.

If we assume

$$
P(N_n=k)=\binom nk p^kq^{n-k},
$$

then

$$
\begin{aligned}
P(N_{n+1}=k)
&=p\binom n{k-1}p^{k-1}q^{n-k+1}
+q\binom nk p^kq^{n-k} \\
&=\left[\binom n{k-1}+\binom nk\right]p^kq^{n+1-k} \\
&=\binom{n+1}{k}p^kq^{n+1-k}.
\end{aligned}
$$

The last step is Pascal's identity. This proves the formula by induction.

In [ ]:
p = 0.4
n = 12
ks = np.arange(n+1)
pmf = np.array([binom_pmf(n, int(k), p) for k in ks])

plt.figure(figsize=(7,4))
plt.bar(ks, pmf)
plt.xlabel('k')
plt.ylabel('P(N_n = k)')
plt.title(f'Binomial distribution: n={n}, p={p}')
plt.show()

pmf.sum()

### Example from the chapter

Using the binomial formula:

$$
P(N_5=4)=\binom54p^4q = 5p^4q.
$$

Also,

$$
P(N_3-N_7=3)
$$

is usually written with the later time first; the intended increment is

$$
P(N_7-N_3=3)=\binom43p^3q = 4p^3q.
$$

The important idea is that increments over disjoint blocks are binomial.

## 4. Independent and stationary increments

For $m,n\ge 0$,

$$
N_{m+n}-N_m = X_{m+1}+\cdots+X_{m+n}.
$$

This has the same distribution as $N_n$:

$$
P(N_{m+n}-N_m=k)=\binom nkp^kq^{n-k}.
$$

Also, increments over disjoint intervals are independent. For example,

$$
N_5, \quad N_9-N_5, \quad N_{13}-N_9
$$

are independent.

Mental model: the past count tells you how many successes already happened, but it tells you nothing about future independent trials.

### Proof idea

The increment

$$
N_{m+n}-N_m
$$

is a function only of

$$
X_{m+1},\ldots,X_{m+n}.
$$

The past count $N_m$ is a function only of

$$
X_1,\ldots,X_m.
$$

These two collections of Bernoulli trials are independent, so the functions of them are independent.

Stationarity comes from the fact that every block of $n$ trials has the same joint distribution.

In [ ]:
p = 0.35
paths = 200_000
X = simulate_bernoulli(p, 13, paths)
N = np.cumsum(X, axis=1)
N5 = N[:,4]
inc_5_9 = N[:,8] - N[:,4]
inc_9_13 = N[:,12] - N[:,8]

# Empirical correlations should be close to 0 for independent increments.
np.corrcoef(np.vstack([N5, inc_5_9, inc_9_13]))

### Example: a compound probability

Suppose we want

$$
P(N_5=4,\; N_7=5,\; N_{13}=8).
$$

Rewrite the event in terms of independent increments:

$$
\{N_5=4,\; N_7-N_5=1,\; N_{13}-N_7=3\}.
$$

Therefore

$$
\begin{aligned}
P(N_5=4, N_7=5, N_{13}=8)
&=P(N_5=4)P(N_7-N_5=1)P(N_{13}-N_7=3)\\
&=\binom54p^4q\binom21pq\binom63p^3q^3\\
&=200p^8q^5.
\end{aligned}
$$

In [ ]:
p = 0.4
q = 1-p
formula = 200 * p**8 * q**5
paths = 2_000_000
X = simulate_bernoulli(p, 13, paths)
N = np.cumsum(X, axis=1)
event = (N[:,4] == 4) & (N[:,6] == 5) & (N[:,12] == 8)
event.mean(), formula

## 5. Conditional expectation in a Bernoulli process

A recurring theme in the chapter is:

> Given the present count, the future behaves like a fresh Bernoulli process started from that count.

For example,

$$
E[N_{11}\mid N_5]
= E[N_5 + (N_{11}-N_5)\mid N_5].
$$

Since $N_{11}-N_5$ is independent of $N_5$ and has expectation $6p$,

$$
E[N_{11}\mid N_5]=N_5+6p.
$$

In [ ]:
p = 0.4
paths = 500_000
X = simulate_bernoulli(p, 11, paths)
N = np.cumsum(X, axis=1)
N5 = N[:,4]
N11 = N[:,10]

# Empirical E[N11 | N5 = j] versus j + 6p
for j in range(6):
    mask = (N5 == j)
    if mask.sum() > 0:
        print(j, N11[mask].mean(), j + 6*p)

### Example: nested conditional expectations

The chapter computes examples such as

$$
E[N_3N_7\mid N_5].
$$

Rewrite

$$
N_7=N_5+(N_7-N_5).
$$

Since $N_7-N_5$ is independent of the past and has expectation $2p$,

$$
E[N_7\mid N_5]=N_5+2p.
$$

Because $N_3$ is determined once the past up to time $5$ is known, conditioning on enough past information gives

$$
E[N_3N_7\mid N_5, N_3] = N_3(N_5+2p).
$$

The book uses this kind of step repeatedly: split future quantities into known present plus independent future increments.

## 6. Times of successes

Define $T_k$ to be the trial number at which the $k$th success occurs.

For example, for the path

$$
0,1,0,1,1,0,\ldots
$$

we have

$$
T_1=2, \quad T_2=4, \quad T_3=5.
$$

There is a direct relation between $N_n$ and $T_k$:

$$
\{T_k\le n\}=\{N_n\ge k\}.
$$

Also,

$$
\{T_k=n\}=\{N_{n-1}=k-1,\; X_n=1\}.
$$

### Distribution of $T_k$

For $n=k,k+1,\ldots$,

$$
P(T_k=n)=\binom{n-1}{k-1}p^kq^{n-k}.
$$

This is the **negative binomial distribution** in the form “time of the $k$th success.”

Proof:

To have the $k$th success exactly at time $n$:

1. trial $n$ must be a success;
2. among the first $n-1$ trials there must be exactly $k-1$ successes.

Thus

$$
P(T_k=n)
=P(N_{n-1}=k-1)P(X_n=1)
=\binom{n-1}{k-1}p^{k-1}q^{n-k}p.
$$

In [ ]:
p = 0.3
k = 4
ns = np.arange(k, 40)
pmf = np.array([math.comb(n-1, k-1)*p**k*(1-p)**(n-k) for n in ns])

plt.figure(figsize=(7,4))
plt.bar(ns, pmf)
plt.xlabel('n')
plt.ylabel('P(T_k = n)')
plt.title(f'Distribution of time T_{k} of the {k}th success, p={p}')
plt.show()

pmf.sum()  # truncated mass

## 7. Waiting times between successes

Define the interarrival/waiting times

$$
W_1=T_1,
$$

and for $k\ge 1$,

$$
W_{k+1}=T_{k+1}-T_k.
$$

Then

$$
P(W_k=m)=q^{m-1}p, \qquad m=1,2,\ldots
$$

So the waiting times are independent and identically distributed geometric random variables.

Mental model: after each success, the process starts over. The future trials do not remember how long it took to get the previous success.

### Proof of the geometric law

For the next success to occur after exactly $m$ trials, we need:

$$
0,0,\ldots,0,1
$$

with $m-1$ failures followed by one success. Therefore

$$
P(W=m)=q^{m-1}p.
$$

The independence of waiting times follows from independence of disjoint blocks of trials.

In [ ]:
p = 0.25
m = np.arange(1, 25)
pmf = np.array([geom_pmf(int(i), p) for i in m])

plt.figure(figsize=(7,4))
plt.bar(m, pmf)
plt.xlabel('m')
plt.ylabel('P(W=m)')
plt.title(f'Geometric waiting time, p={p}')
plt.show()

pmf[:10]

### Mean and variance of geometric waiting time

For $W\sim\operatorname{Geom}(p)$ on $\{1,2,\ldots\}$,

$$
E[W]=\sum_{m=1}^\infty m q^{m-1}p = \frac1p.
$$

Also,

$$
\operatorname{Var}(W)=\frac{q}{p^2}.
$$

Since

$$
T_k=W_1+\cdots+W_k,
$$

we get

$$
E[T_k]=\frac{k}{p},
$$

and

$$
\operatorname{Var}(T_k)=\frac{kq}{p^2}.
$$

In [ ]:
p = 0.2
k = 10
paths = 200_000
# Simulate T_k by summing k geometric waiting times.
W = rng.geometric(p, size=(paths, k))
Tk = W.sum(axis=1)
Tk.mean(), k/p, Tk.var(), k*(1-p)/p**2

### Example: exact probability with success times

Compute

$$
P(T_1=3,\; T_3=9,\; T_7=17).
$$

Rewrite in waiting times:

$$
T_1=3,
$$

$$
T_3-T_1=6,
$$

$$
T_7-T_3=8.
$$

The first means the first success arrives at trial $3$:

$$
P(T_1=3)=q^2p.
$$

The second means the next two successes take $6$ trials total. This is the time of the second success in a fresh process:

$$
P(T_2=6)=\binom51p^2q^4.
$$

The third means the next four successes take $8$ trials total:

$$
P(T_4=8)=\binom73p^4q^4.
$$

Therefore

$$
P(T_1=3,T_3=9,T_7=17)
= (q^2p)\left(\binom51p^2q^4\right)\left(\binom73p^4q^4\right).
$$

In [ ]:
p = 0.4
q = 1-p
formula = (q**2*p) * (math.comb(5,1)*p**2*q**4) * (math.comb(7,3)*p**4*q**4)
formula

### Example: expected future failure time

Suppose component failures occur at success times of a Bernoulli process. If we observe

$$
T_1=3,\quad T_2=12,\quad T_3=14,
$$

then the expected time of the fifth failure is

$$
E[T_5\mid T_1,T_2,T_3]
= T_3 + E[(T_4-T_3)+(T_5-T_4)].
$$

Each future waiting time has mean $1/p$, so

$$
E[T_5\mid T_1=3,T_2=12,T_3=14]
=14+\frac{2}{p}.
$$

For $p=0.60$, this is

$$
14+\frac{2}{0.60}=17.33\ldots
$$

In [ ]:
p = 0.60
14 + 2/p

### Example: discounted replacement cost

Suppose each item replacement costs $c$ dollars, and future costs are discounted by a factor $\alpha\in(0,1)$ per time step.

If replacements occur at times $T_1,T_2,\ldots$, the total discounted cost is

$$
C = \sum_{k=1}^\infty c\alpha^{T_k}.
$$

Since

$$
T_k=W_1+\cdots+W_k,
$$

and the $W_i$ are independent geometric random variables,

$$
E[\alpha^{T_k}]
= \left(E[\alpha^W]\right)^k.
$$

For $W\sim\operatorname{Geom}(p)$,

$$
E[\alpha^W]
=\sum_{m=1}^\infty \alpha^m q^{m-1}p
=\frac{\alpha p}{1-\alpha q}.
$$

Therefore

$$
E[C]
= c\sum_{k=1}^\infty\left(\frac{\alpha p}{1-\alpha q}\right)^k
= c\frac{\alpha p}{1-\alpha}.
$$

The simplification occurs because

$$
1-\frac{\alpha p}{1-\alpha q}
=\frac{1-\alpha}{1-\alpha q}.
$$

In [ ]:
p = 0.3
alpha = 0.95
c = 10
expected_cost = c * alpha*p/(1-alpha)
expected_cost

## 8. A useful identity for sums over success times

For a nonnegative function $f$,

$$
E\left[\sum_{n=1}^\infty f(T_n)\right]
= p\sum_{n=1}^\infty f(n).
$$

Reason:

A time $m$ appears among the success times $T_1,T_2,\ldots$ exactly when $X_m=1$. Therefore

$$
\sum_{n=1}^\infty f(T_n)
=\sum_{m=1}^\infty f(m)X_m.
$$

Taking expectations gives

$$
E\left[\sum_{m=1}^\infty f(m)X_m\right]
=\sum_{m=1}^\infty f(m)E[X_m]
=p\sum_{m=1}^\infty f(m).
$$

In [ ]:
# Example: f(n)=exp(-lambda n)
p = 0.4
lam = 0.2
M = 10_000
lhs_theory = p * sum(math.exp(-lam*n) for n in range(1, M))
rhs_closed = p * math.exp(-lam)/(1 - math.exp(-lam))
lhs_theory, rhs_closed

## 9. Sums of independent random variables

Now let

$$
Y_1,Y_2,\ldots
$$

be independent identically distributed random variables, and define

$$
Z_0=0,
$$

$$
Z_n=Y_1+\cdots+Y_n,\qquad n\ge 1.
$$

This generalizes $N_n$, where $Y_i=X_i$ are Bernoulli.

The process $Z_n$ has **stationary independent increments**:

- $Z_{n+m}-Z_n$ has the same distribution as $Z_m$;
- increments over disjoint intervals are independent.

### Expectation and variance

If

$$
E[Y_i]=a,
$$

and

$$
\operatorname{Var}(Y_i)=b^2,
$$

then

$$
E[Z_n]=na,
$$

and, by independence,

$$
\operatorname{Var}(Z_n)=nb^2.
$$

Therefore the sample average

$$
\frac{Z_n}{n}
$$

has

$$
E\left[\frac{Z_n}{n}\right]=a,
$$

and

$$
\operatorname{Var}\left(\frac{Z_n}{n}\right)=\frac{b^2}{n}.
$$

This is the quantitative reason averages stabilize.

In [ ]:
a = 4.87
b = 2.16
n_values = np.array([5, 10, 20, 50, 100, 500, 1000])
var_avg = b**2 / n_values
list(zip(n_values, var_avg))

### Weak law of large numbers

By Chebyshev's inequality,

$$
P\left(\left|\frac{Z_n}{n}-a\right|>\varepsilon\right)
\le \frac{b^2}{n\varepsilon^2}.
$$

As $n\to\infty$, the right-hand side goes to $0$. Hence

$$
\lim_{n\to\infty}P\left(\left|\frac{Z_n}{n}-a\right|>\varepsilon\right)=0.
$$

This is the **weak law of large numbers**.

Interpretation: for large $n$, the average is probably close to the mean.

In [ ]:
# Simulate sample averages to see the weak law.
a = 4.87
sigma = 2.16
paths = 20_000
max_n = 1000
Y = rng.normal(a, sigma, size=(paths, max_n))
Z = np.cumsum(Y, axis=1)
for n in [5, 20, 100, 1000]:
    avg = Z[:, n-1] / n
    print(n, avg.mean(), avg.var(), np.mean(np.abs(avg-a) > 0.5))

In [ ]:
# Plot several sample average paths.
a = 4.87
sigma = 2.16
paths = 8
max_n = 500
Y = rng.normal(a, sigma, size=(paths, max_n))
Z = np.cumsum(Y, axis=1)
A = Z / np.arange(1, max_n+1)

plt.figure(figsize=(8,4))
for i in range(paths):
    plt.plot(np.arange(1, max_n+1), A[i], linewidth=1)
plt.axhline(a, linestyle='--')
plt.xlabel('n')
plt.ylabel('sample average Z_n/n')
plt.title('Sample averages stabilize near the mean')
plt.show()

### Strong law of large numbers

The chapter also states the stronger result:

$$
P\left(\lim_{n\to\infty}\frac{Z_n}{n}=a\right)=1.
$$

This is the **strong law of large numbers**.

Difference between weak and strong laws:

- Weak law: for a fixed large $n$, the average is probably close to $a$.
- Strong law: along almost every infinite sample path, the average converges to $a$.

Mental model: weak law is about the distribution at time $n$; strong law is about long-run behavior of individual paths.

## 10. Connecting the chapter's main ideas

The chapter builds one ladder:

1. Start with independent Bernoulli trials $X_n$.
2. Count successes: $N_n=X_1+\cdots+X_n$.
3. Derive the binomial law for $N_n$.
4. Use independent increments to compute compound probabilities and conditional expectations.
5. Study success times $T_k$.
6. Show waiting times $T_{k+1}-T_k$ are i.i.d. geometric.
7. Generalize from Bernoulli sums to arbitrary sums $Z_n=Y_1+\cdots+Y_n$.
8. Use expectation, variance, and Chebyshev to prove the weak law.
9. State the strong law as the pathwise version.

The core thinking model is:

> Break the object into independent increments. Then compute with expectation, variance, or products of probabilities.

## 11. Quick reference

### Bernoulli trial

$$
P(X=1)=p, \qquad P(X=0)=q=1-p.
$$

$$
E[X]=p, \qquad \operatorname{Var}(X)=pq.
$$

### Number of successes

$$
N_n=X_1+\cdots+X_n.
$$

$$
P(N_n=k)=\binom nkp^kq^{n-k}.
$$

$$
E[N_n]=np, \qquad \operatorname{Var}(N_n)=npq.
$$

### Increments

$$
N_{m+n}-N_m\sim\operatorname{Binomial}(n,p),
$$

independent of the past.

### Success time

$$
P(T_k=n)=\binom{n-1}{k-1}p^kq^{n-k},\qquad n=k,k+1,\ldots
$$

### Waiting time

$$
P(T_{k+1}-T_k=m)=q^{m-1}p,\qquad m=1,2,\ldots
$$

$$
E[T_k]=\frac{k}{p}, \qquad \operatorname{Var}(T_k)=\frac{kq}{p^2}.
$$

### General sums

$$
Z_n=Y_1+\cdots+Y_n.
$$

If $E[Y_i]=a$ and $\operatorname{Var}(Y_i)=b^2$, then

$$
E[Z_n]=na, \qquad \operatorname{Var}(Z_n)=nb^2.
$$

Weak law:

$$
\frac{Z_n}{n}\to a \quad \text{in probability}.
$$

Strong law:

$$
\frac{Z_n}{n}\to a \quad \text{almost surely}.
$$